# TikzTable: basics

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

`TikzTable` shares its entire builder API with [`TexTable`](textable.ipynb): every call on
that page (index columns, rules, header groups, highlights, formatters) works here,
unchanged, because both classes are emitters over one shared implementation. What
changes is the TeX dialect: `TikzTable` renders through `nicematrix`, which lays the
table out on a TikZ cell-position lattice.

That lattice is the reason to choose it: per-cell borders, free-form `\draw` commands,
and cell-geometry control that a plain `tabular` cannot express (see
[TikzTable styling](tikztable_styling.ipynb)). The cost is a heavier preamble
(`nicematrix` and `tikz`) and **two** compiler passes, because `nicematrix` records
cell positions on the first pass and draws on the second. `preview()`, `save_pdf()`,
and `save_png()` handle the double pass; if the table does not need those drawing features,
prefer `TexTable`.

In [ ]:
import numpy as np
import pandas as pd

districts = pd.DataFrame(
    {
        "District": [f"CD {i}" for i in range(1, 9)],
        "BVAP share": [0.12, 0.18, 0.22, 0.31, 0.38, 0.44, 0.52, 0.58],
        "Dem share": [0.35, 0.41, 0.44, 0.47, 0.50, 0.55, 0.61, 0.66],
        "Polsby-Popper": [0.18, 0.22, 0.27, 0.31, 0.33, 0.35, 0.41, 0.44],
        "Pop. deviation": [0.004, 0.002, np.nan, 0.006, 0.001, 0.008, 0.003, 0.005],
    }
)
districts

## The default table

In [ ]:
from gerrytools.latex import TikzTable

table = TikzTable(districts)
table.set_decimal_count(3)
print(table)

![Default TikzTable][tikztable-default]

[tikztable-default]: ../../_static/images/latex/tikztable-default.png

The printed source is again a complete standalone document; note the `nicematrix`
preamble and the `NiceTabular` environment. `print_table()` prints the environment
alone.

## Index columns and header groups

In [ ]:
table = TikzTable(districts)
table.set_decimal_count(2)
table.include_index(name="Row")
table.set_header_groups(
    {
        "Identity": ["District"],
        "Demographics and votes": ["BVAP share", "Dem share"],
        "Diagnostics": ["Polsby-Popper", "Pop. deviation"],
    }
)
print(table)

![TikzTable with index and grouped headers][tikztable-groups]

[tikztable-groups]: ../../_static/images/latex/tikztable-groups.png

## Row highlights

In [ ]:
table = TikzTable(districts)
table.set_decimal_count(2)
table.highlight_rows([1], color="cherryblossompink")
table.highlight_rows([4], color="lightblue!35!white")
table.highlight_rows([7], color="amber!40!white")
print(table)

![Highlighted rows in a TikzTable][tikztable-highlights]

[tikztable-highlights]: ../../_static/images/latex/tikztable-highlights.png

## Rules

The rule controls match `TexTable`, plus `add_toprule()` / `add_bottomrule()` for
booktabs-style framing.

In [ ]:
table = TikzTable(districts)
table.set_decimal_count(2)
table.add_vrule_right_of(0)
table.add_hrule_above([4])
table.add_toprule()
table.add_bottomrule()
print(table)

![TikzTable with top and bottom rules][tikztable-rules]

[tikztable-rules]: ../../_static/images/latex/tikztable-rules.png

## Related

- [TikzTable styling](tikztable_styling.ipynb) for borders, gradients, and
  drawing
- [TexTable](textable.ipynb) for the plain-tabular variant
- [LaTeX API](../../api/latex.rst)